# LETO initial-state walkthrough

This notebook follows the ArcPy LETO workflow from management units and TreeMap cells through LETO-compatible FVS stand and tree tables. The production transformations live in `pipeline/s1_initial_state`; cells here expose their intermediate diagnostics without reimplementing them.

## Inputs and preflight

In [ ]:
from pathlib import Path
import sys

import geopandas as gpd
import pandas as pd
import rasterio

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'pipeline').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.s1_initial_state.weights import build_plot_weights
from pipeline.s1_initial_state.leto_initial_state import (
    build_initial_state,
    build_management_unit_crosswalk,
    filter_and_normalize_weights,
    load_fia_tree_files,
    load_species_lookup,
    prepare_direct_tree_rows,
    write_initial_state,
)

In [ ]:
MANAGEMENT_UNITS_PATH = PROJECT_ROOT / 'data/interim/management_units_5co/units_with_attributes.gpkg'
MANAGEMENT_UNITS_LAYER = None
TREEMAP_PATH = PROJECT_ROOT / 'data/interim/treemap_2022_fl.tif'
TREEMAP_LOOKUP_PATH = PROJECT_ROOT / 'data/interim/treemap_tmids_fl.csv'
TREEMAP_LOOKUP_VALUE_COLUMN = 'VALUE'
SPECIES_CROSSWALK_PATH = PROJECT_ROOT / 'data/raw/FVS_SpeciesCrosswalk.xls'
SPECIES_CROSSWALK_SHEET = 'EasternSpeciesTranslator'
FIA_TREE_PATHS = [PROJECT_ROOT / 'data/raw/fia/FL_TREE.csv']
OUTPUT_DIR = PROJECT_ROOT / 'data/interim/fvs/leto_initial_state'
MIN_PLT_WEIGHT = 0.05
WRITE_OUTPUTS = False

inputs = {
    'management units': MANAGEMENT_UNITS_PATH,
    'TreeMap raster': TREEMAP_PATH,
    'TreeMap lookup': TREEMAP_LOOKUP_PATH,
    'species crosswalk': SPECIES_CROSSWALK_PATH,
    **{f'FIA TREE {index + 1}': path for index, path in enumerate(FIA_TREE_PATHS)},
}
preflight = pd.DataFrame(
    [{'input': name, 'path': str(path), 'exists': path.exists()} for name, path in inputs.items()]
)
display(preflight)
missing_inputs = preflight.loc[~preflight['exists'], 'path'].tolist()
if missing_inputs:
    raise FileNotFoundError('Update the input paths before continuing: ' + ', '.join(missing_inputs))

## Management units and TreeMap alignment

LETO rasterized `MU_ID` on the TreeMap grid. The Python port reads only the raster window covering these units and uses the portable pixel-center rule.

In [ ]:
management_units = gpd.read_file(MANAGEMENT_UNITS_PATH, layer=MANAGEMENT_UNITS_LAYER)
with rasterio.open(TREEMAP_PATH) as source:
    treemap_metadata = pd.Series({
        'crs': str(source.crs),
        'pixel_width': source.transform.a,
        'pixel_height': source.transform.e,
        'nodata': source.nodata,
        'width': source.width,
        'height': source.height,
    })
display(management_units.drop(columns='geometry').head())
display(treemap_metadata)
management_units.plot(column='MU_ID', figsize=(8, 8), legend=False)

## MU x PLT_CN weights

This is the non-ArcPy counterpart to `assign_plt_cn` in `LETO.V1.1.txt`.

In [ ]:
treemap_lookup = pd.read_csv(TREEMAP_LOOKUP_PATH, dtype={'PLT_CN': 'string'})
plot_weights = build_plot_weights(
    management_units,
    TREEMAP_PATH,
    treemap_lookup,
    lookup_value_column=TREEMAP_LOOKUP_VALUE_COLUMN,
)
weight_diagnostics = plot_weights.groupby('MU_ID').agg(
    donor_plots=('PLT_CN', 'size'),
    valid_cells=('CELL_COUNT', 'sum'),
    raw_weight_sum=('WEIGHT', 'sum'),
)
display(plot_weights.head())
display(weight_diagnostics.describe())

## FIA join coverage

LETO filters donors below five percent, renormalizes within each management unit, and joins multistate FIA tree files because TreeMap can borrow plots across state lines.

In [ ]:
crosswalk = build_management_unit_crosswalk(management_units, plot_weights)
normalized_weights = filter_and_normalize_weights(
    plot_weights, crosswalk, min_plot_weight=MIN_PLT_WEIGHT
)
fia_trees = load_fia_tree_files(FIA_TREE_PATHS)
weighted_plots = set(normalized_weights['PLT_CN'])
fia_plots = set(fia_trees['PLT_CN'])
join_coverage = pd.Series({
    'weighted plots': len(weighted_plots),
    'matched FIA plots': len(weighted_plots & fia_plots),
    'unmatched FIA plots': len(weighted_plots - fia_plots),
})
display(join_coverage)
display(pd.Series(sorted(weighted_plots - fia_plots), name='unmatched_PLT_CN').head(25))

## Species and live-tree preparation

In [ ]:
species_lookup = load_species_lookup(SPECIES_CROSSWALK_PATH, SPECIES_CROSSWALK_SHEET)
direct_trees = prepare_direct_tree_rows(normalized_weights, fia_trees, species_lookup)
tree_diagnostics = pd.Series({
    'combined FIA rows': len(fia_trees),
    'direct FVS tree rows': len(direct_trees),
    'direct runnable stands': direct_trees['STAND_ID'].nunique(),
    'translated FVS species': direct_trees['SPECIES'].nunique(),
})
display(tree_diagnostics)
display(direct_trees.head())

## Nearest-runnable-unit imputation

Units without usable live trees receive the nearest runnable polygon's weighted tree list. `TREE_SOURCE`, `DONOR_STAND_ID`, and `NEAR_DIST` preserve that provenance.

In [ ]:
tables = build_initial_state(
    management_units,
    plot_weights,
    fia_trees,
    species_lookup,
    min_plot_weight=MIN_PLT_WEIGHT,
)
display(tables.diagnostics.to_frame())
source_summary = tables.trees.groupby('TREE_SOURCE').agg(
    tree_rows=('TREE_ID', 'size'),
    stands=('STAND_ID', 'nunique'),
)
display(source_summary)
display(tables.trees.loc[tables.trees['TREE_SOURCE'] == 'IMPUTED_NEAREST', [
    'STAND_ID', 'DONOR_STAND_ID', 'NEAR_DIST'
]].drop_duplicates().head(25))

## Initial-state map

In [ ]:
direct_ids = set(tables.trees.loc[tables.trees['TREE_SOURCE'] == 'FIA_WEIGHTED_DIRECT', 'MU_ID'])
imputed_ids = set(tables.trees.loc[tables.trees['TREE_SOURCE'] == 'IMPUTED_NEAREST', 'MU_ID'])
state_map = management_units.copy()
state_map['MU_ID'] = state_map['MU_ID'].astype('string')
state_map['INITIAL_STATE'] = 'missing'
state_map.loc[state_map['MU_ID'].isin(imputed_ids), 'INITIAL_STATE'] = 'imputed_nearest'
state_map.loc[state_map['MU_ID'].isin(direct_ids), 'INITIAL_STATE'] = 'fia_weighted_direct'
display(state_map['INITIAL_STATE'].value_counts())
state_map.plot(column='INITIAL_STATE', categorical=True, legend=True, figsize=(10, 10))

## Write LETO-compatible outputs

In [ ]:
if WRITE_OUTPUTS:
    output_paths = write_initial_state(tables, OUTPUT_DIR)
    display(pd.Series({name: str(path) for name, path in output_paths.items()}))
else:
    print('WRITE_OUTPUTS is False; inspected results were not written.')